# CLA 1 - Curriculum-Industry Skill Feature Store Using Feast

This notebook uses the supplied **100-row `skill_gap_dataset_100.csv`** and demonstrates the complete Feast workflow required by the assignment:

1. Feature engineering
2. Feast entity creation
3. Feast data source creation
4. FeatureView creation
5. `feast apply`
6. Historical feature retrieval
7. Materialization into the online store
8. Online feature retrieval
9. Machine-learning model
10. Results


In [ ]:
# STEP 1: Upload your dataset to Colab
from google.colab import files
uploaded = files.upload()

import os
filename = list(uploaded.keys())[0]
print('Uploaded:', filename)


Saving skill_gap_dataset_100.csv to skill_gap_dataset_100.csv
Uploaded: skill_gap_dataset_100.csv


In [ ]:
# STEP 2: Install required packages
!pip -q install feast pandas pyarrow scikit-learn
print('Packages installed.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.0 requires tenacity<10,>=9, but you have tenacity 8.5.0 which is incompatible.
Packages installed.


In [ ]:
# STEP 3: Read and inspect the supplied dataset
import pandas as pd

df = pd.read_csv(filename)
print('Shape:', df.shape)
display(df.head(10))



Shape: (100, 6)


,Skill_Name,Curriculum_Coverage,Industry_Demand,Job_Frequency,Expert_Importance,Gap_Status
0,Spring Boot_1,0,0,4,1,Aligned
1,ExpressJS_2,5,0,10,5,Aligned
2,Cyber Security_3,0,4,6,0,Aligned
3,React_4,1,1,8,4,Aligned
4,DevOps_5,1,5,10,5,Gap
5,Cyber Security_6,3,1,7,4,Aligned
6,AWS_7,0,1,6,2,Gap
7,MongoDB_8,1,2,1,0,Gap
8,Angular_9,2,2,9,2,Aligned
9,Problem Solving_10,3,4,1,3,Aligned


In [ ]:
# STEP 4: Feature engineering
import numpy as np
from pathlib import Path

PROJECT = Path('/content/skill_gap_feast')
DATA = PROJECT / 'data'
DATA.mkdir(parents=True, exist_ok=True)

# Preserve the original supplied dataset
df.to_csv(DATA / 'skill_gap_dataset_100.csv', index=False)

features = df.copy()
features.insert(0, 'skill_id', [f'S{i:03d}' for i in range(1, len(features) + 1)])
features['event_timestamp'] = pd.Timestamp('2026-08-01', tz='UTC')

features['skill_gap'] = features['Industry_Demand'] - features['Curriculum_Coverage']
features['demand_ratio'] = np.where(
    features['Curriculum_Coverage'] == 0,
    features['Industry_Demand'].astype(float),
    features['Industry_Demand'] / features['Curriculum_Coverage']
).round(3)
features['high_gap'] = (features['Gap_Status'].str.strip().str.lower() == 'gap').astype(int)

features.to_parquet(DATA / 'skill_gap_features.parquet', index=False)

print('Feature dataset created:', features.shape)
display(features.head(10))


Feature dataset created: (100, 11)


,skill_id,Skill_Name,Curriculum_Coverage,Industry_Demand,Job_Frequency,Expert_Importance,Gap_Status,event_timestamp,skill_gap,demand_ratio,high_gap
0,S001,Spring Boot_1,0,0,4,1,Aligned,2026-08-01 00:00:00+00:00,0,0.000,0
1,S002,ExpressJS_2,5,0,10,5,Aligned,2026-08-01 00:00:00+00:00,-5,0.000,0
2,S003,Cyber Security_3,0,4,6,0,Aligned,2026-08-01 00:00:00+00:00,4,4.000,0
3,S004,React_4,1,1,8,4,Aligned,2026-08-01 00:00:00+00:00,0,1.000,0
4,S005,DevOps_5,1,5,10,5,Gap,2026-08-01 00:00:00+00:00,4,5.000,1
5,S006,Cyber Security_6,3,1,7,4,Aligned,2026-08-01 00:00:00+00:00,-2,0.333,0
6,S007,AWS_7,0,1,6,2,Gap,2026-08-01 00:00:00+00:00,1,1.000,1
7,S008,MongoDB_8,1,2,1,0,Gap,2026-08-01 00:00:00+00:00,1,2.000,1
8,S009,Angular_9,2,2,9,2,Aligned,2026-08-01 00:00:00+00:00,0,1.000,0
9,S010,Problem Solving_10,3,4,1,3,Aligned,2026-08-01 00:00:00+00:00,1,1.333,0


In [ ]:
# STEP 5: Create Feast configuration
feature_store_yaml = '''project: skill_gap_feast
provider: local
registry: data/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
entity_key_serialization_version: 2
'''

(PROJECT / 'feature_store.yaml').write_text(feature_store_yaml)
print(feature_store_yaml)


project: skill_gap_feast
provider: local
registry: data/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
entity_key_serialization_version: 2



In [ ]:
# STEP 6: Create Feast entity, data source and FeatureView
feature_store_py = '''from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

skill = Entity(
    name='skill_id',
    join_keys=['skill_id'],
    description='Unique identifier for each skill in the supplied dataset.'
)

source = FileSource(
    name='skill_gap_source',
    path='../data/skill_gap_features.parquet',
    timestamp_field='event_timestamp'
)

skill_gap_features = FeatureView(
    name='skill_gap_features',
    entities=[skill],
    ttl=timedelta(days=365),
    schema=[
        Field(name='Curriculum_Coverage', dtype=Int64),
        Field(name='Industry_Demand', dtype=Int64),
        Field(name='Job_Frequency', dtype=Int64),
        Field(name='Expert_Importance', dtype=Int64),
        Field(name='skill_gap', dtype=Int64),
        Field(name='demand_ratio', dtype=Float32),
        Field(name='high_gap', dtype=Int64),
    ],
    source=source,
    online=True,
)
'''

(PROJECT / 'feature_store.py').write_text(feature_store_py)
print(feature_store_py)


from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

skill = Entity(
    name='skill_id',
    join_keys=['skill_id'],
    description='Unique identifier for each skill in the supplied dataset.'
)

source = FileSource(
    name='skill_gap_source',
    path='../data/skill_gap_features.parquet',
    timestamp_field='event_timestamp'
)

skill_gap_features = FeatureView(
    name='skill_gap_features',
    entities=[skill],
    ttl=timedelta(days=365),
    schema=[
        Field(name='Curriculum_Coverage', dtype=Int64),
        Field(name='Industry_Demand', dtype=Int64),
        Field(name='Job_Frequency', dtype=Int64),
        Field(name='Expert_Importance', dtype=Int64),
        Field(name='skill_gap', dtype=Int64),
        Field(name='demand_ratio', dtype=Float32),
        Field(name='high_gap', dtype=Int64),
    ],
    source=source,
    online=True,
)



In [ ]:
# STEP 7: Register the Feast objects with feast apply
%cd /content/skill_gap_feast
!feast apply


/content/skill_gap_feast
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib

In [ ]:
from feast import FeatureStore
from pathlib import Path
import os

# Fix for the FileNotFoundError in feast apply (from previous steps)
# The original feature_store.py had an incorrect path for FileSource.
# It was '../data/skill_gap_features.parquet', but it should be 'data/skill_gap_features.parquet'
# when feast apply is run from /content/skill_gap_feast.
PROJECT = Path('/content/skill_gap_feast')
feature_store_py_fixed = '''from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

skill = Entity(
    name='skill_id',
    join_keys=['skill_id'],
    description='Unique identifier for each skill in the supplied dataset.'
)

source = FileSource(
    name='skill_gap_source',
    path='data/skill_gap_features.parquet', # CORRECTED PATH HERE
    timestamp_field='event_timestamp'
)

skill_gap_features = FeatureView(
    name='skill_gap_features',
    entities=[skill],
    ttl=timedelta(days=365),
    schema=[
        Field(name='Curriculum_Coverage', dtype=Int64),
        Field(name='Industry_Demand', dtype=Int64),
        Field(name='Job_Frequency', dtype=Int64),
        Field(name='Expert_Importance', dtype=Int64),
        Field(name='skill_gap', dtype=Int64),
        Field(name='demand_ratio', dtype=Float32),
        Field(name='high_gap', dtype=Int64),
    ],
    source=source,
    online=True,
)
'''

(PROJECT / 'feature_store.py').write_text(feature_store_py_fixed)
print("Corrected feature_store.py and re-applying Feast configuration...")

# Re-run feast apply after correcting the feature_store.py
# Use '%%capture' to suppress feast apply output if desired, or let it print.
original_dir = os.getcwd()
os.chdir(PROJECT)
!feast apply
os.chdir(original_dir)
print("Feast apply re-run successfully.")

store = FeatureStore(repo_path='/content/skill_gap_feast')

feature_refs = [
    'skill_gap_features:Curriculum_Coverage',
    'skill_gap_features:Industry_Demand',
    'skill_gap_features:Job_Frequency',
    'skill_gap_features:Expert_Importance',
    'skill_gap_features:skill_gap',
    'skill_gap_features:demand_ratio',
]

# `features` variable is available from previous cells in the kernel state.
entity_df = features[['skill_id', 'event_timestamp', 'high_gap']].copy()

historical = store.get_historical_features(
    entity_df=entity_df,
    features=feature_refs,
).to_df()

display(historical.head(10))

Corrected feature_store.py and re-applying Feast configuration...
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/d

/usr/local/lib/python3.12/dist-packages/feast/repo_config.py:459: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,skill_id,event_timestamp,high_gap,Curriculum_Coverage,Industry_Demand,Job_Frequency,Expert_Importance,skill_gap,demand_ratio
0,S001,2026-08-01 00:00:00+00:00,0,0,0,4,1,0,0.000
1,S073,2026-08-01 00:00:00+00:00,1,5,5,4,4,0,1.000
2,S072,2026-08-01 00:00:00+00:00,1,2,4,3,5,2,2.000
3,S071,2026-08-01 00:00:00+00:00,0,2,0,1,5,-2,0.000
4,S070,2026-08-01 00:00:00+00:00,0,4,2,10,0,-2,0.500
5,S069,2026-08-01 00:00:00+00:00,0,3,4,4,4,1,1.333
6,S068,2026-08-01 00:00:00+00:00,1,2,0,3,2,-2,0.000
7,S067,2026-08-01 00:00:00+00:00,0,4,1,8,2,-3,0.250
8,S066,2026-08-01 00:00:00+00:00,0,0,0,7,4,0,0.000
9,S065,2026-08-01 00:00:00+00:00,1,1,5,10,2,4,5.000


In [ ]:
# STEP 9: Materialize features into the SQLite online store
!feast materialize-incremental 2026-08-17T23:59:59


/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [ ]:
# STEP 10: Online feature retrieval
online = store.get_online_features(
    features=feature_refs,
    entity_rows=[
        {'skill_id': 'S005'},
        {'skill_id': 'S007'},
        {'skill_id': 'S015'},
    ],
).to_dict()

online_df = pd.DataFrame(online)
display(online_df)


/usr/local/lib/python3.12/dist-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


,skill_id,Expert_Importance,Job_Frequency,Curriculum_Coverage,Industry_Demand,demand_ratio,skill_gap
0,S005,5,10,1,5,5.0,4
1,S007,2,6,0,1,1.0,1
2,S015,5,10,2,5,2.5,3


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# STEP 11: Use Feast features in a machine-learning model
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

feature_cols = [
    'Curriculum_Coverage',
    'Industry_Demand',
    'Job_Frequency',
    'Expert_Importance',
    'skill_gap',
    'demand_ratio',
]

X = historical[feature_cols]
y = historical['high_gap']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
accuracy = accuracy_score(y_test, pred)

print(f'Model Accuracy: {accuracy:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, pred))


Model Accuracy: 0.6800

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.69      0.73        16
           1       0.55      0.67      0.60         9

    accuracy                           0.68        25
   macro avg       0.67      0.68      0.67        25
weighted avg       0.70      0.68      0.69        25

Confusion Matrix:
[[11  5]
 [ 3  6]]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# STEP 12: One final prediction using the first real record from your dataset
sample = historical.loc[[0], feature_cols]
final_prediction = int(model.predict(sample)[0])

print('Skill:', features.loc[0, 'Skill_Name'])
print('Actual Gap_Status:', features.loc[0, 'Gap_Status'])
print('Predicted high_gap:', final_prediction)
print('Prediction meaning:', 'Gap' if final_prediction == 1 else 'Aligned')


Skill: Spring Boot_1
Actual Gap_Status: Aligned
Predicted high_gap: 1
Prediction meaning: Gap


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Required Analysis

**1. Entity:** `skill_id`, a generated stable ID for each supplied record.

**2. FeatureView features:** `Curriculum_Coverage`, `Industry_Demand`, `Job_Frequency`, `Expert_Importance`, `skill_gap`, `demand_ratio`, and `high_gap`.

**3. Feature calculation:** `skill_gap = Industry_Demand - Curriculum_Coverage`.

**4. Original vs feature dataset:** The feature dataset retains the supplied values and adds `skill_id`, `event_timestamp`, `skill_gap`, `demand_ratio`, and encoded `high_gap`.

**5. Offline store:** Stores historical feature data for training and historical retrieval.

**6. Online store:** Stores the latest materialized values for fast online prediction.

**7. `feast apply`:** Registers/updates the Feast entities, source and FeatureView.

**8. Materialization:** Copies feature values from the offline source into the online store.

**9. Advantage of Feast:** One centralized feature definition can be reused for historical training and online serving, reducing duplicated feature logic and training-serving inconsistency.

**10. Limitations:** The dataset has only 100 records and does not contain detailed evidence such as job-posting text, company information, dates, salary data, or source URLs.

**11. Improvements:** Add more time-stamped curriculum/job-market evidence and add time-varying features such as job-posting trends, salary demand, emerging skills, and curriculum update frequency.
